# Product Quantization and Asymmetric Distance Computation — companion notebook

The **narrative** half of the Python pillar. The tested implementation lives next door in
[`product_quantization.py`](product_quantization.py); here we import it and walk the topic
section by section, so every claim renders as executed output.

Product quantization **is** Lloyd's k-means run independently per subspace — so the per-subspace
quantizer, the empty-cell repair, the synthetic finance cloud, and the `kmeans2` cross-check are
all imported from the previous topic. `product_quantization.py` owns the numbers; its `test_*`
assertions encode the theorems and `viz_constants()` prints what the laboratory mirrors.

In [ ]:
import sys, pathlib
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "product-quantization",
              pathlib.Path("notebooks/product-quantization")):
    if (_cand / "product_quantization.py").exists():
        sys.path.insert(0, str(_cand)); break
import numpy as np
import product_quantization as pq

## 1. The product quantizer and the additive decomposition

Split `x ∈ R^d` into `m` disjoint blocks, quantize each with its own codebook of `k*` centroids,
and store the tuple of `m` indices (`m·log₂k*` bits). Because squared Euclidean distance separates
over disjoint coordinate blocks, the total distortion is the **sum** of the per-subspace
distortions — so each subspace is trained independently by Lloyd (**Theorem 1**).

In [ ]:
X = pq.toy_pq_cloud()
cb = pq.train_pq(X, pq.TOY_M, pq.TOY_KSTAR, seed=0)
sub = pq.pq_subspace_distortions(X, cb)
print(f"toy cloud: {X.shape[0]} points in R^4 = two 2-D subspaces, k*={pq.TOY_KSTAR}")
print(f"per-subspace distortion = {np.round(sub,3).tolist()}  (sum {sub.sum():.3f})")
print(f"total ||x-Q(x)||^2      = {pq.pq_distortion(X, cb):.3f}")
pq.test_additive_decomposition()

## 2. Asymmetric distance is exact — and a table lookup

Keep the query `q` un-quantized. Then `‖q − Q(x)‖² = Σⱼ LUT[j][q_j(x)]` where `LUT[j][i] = ‖q^j − c^j_i‖²`
is an `m×k*` table built once per query (**Theorem 2**). Each database distance is `m` lookups —
`O(m)` instead of `O(d)`. The effective codebook is `(k*)^m` while only `m·k*` centroids are stored.

In [ ]:
pq.test_adc_equals_bruteforce()
pq.test_effective_codebook_size()

## 3. Asymmetric beats symmetric — in the mean

Symmetric distance (SDC) quantizes the query too, adding its quantization error; ADC keeps the
query exact. So ADC's distance-estimate error is lower and its recall higher **on average**
(**Prop 2**) — though individual pairs can reverse, so we assert the mean, not a per-pair claim.

In [ ]:
pq.test_adc_error_below_sdc_error()
pq.test_adc_recall_ge_sdc_recall()

## 4. The honest catch: PQ never beats a *trainable* flat codebook at equal bits

A flat codebook is unconstrained; PQ's reconstruction is confined to the product of its
sub-codebooks. So at any bit budget a flat `k=2^B` codebook can actually train (`k ≤ n`), flat-VQ
distortion is ≤ PQ's. PQ pays a product-separability constraint.

In [ ]:
for r in pq.flat_vs_pq_equal_bits(*pq.finance_dataset()[::2], bits_grid=(4,6,8)):
    print(f"  {r['bits']} bits: flat k={r['k_flat']:4d} D={r['flat_D']:.1f}  |  "
          f"PQ m={r['m']},k*={r['k_star']} D={r['pq_D']:.1f}   (flat <= PQ: {r['flat_D']<=r['pq_D']})")
pq.test_flat_ge_pq_at_equal_bits()

## 5. The headline: scalability reaches what a flat codebook cannot

A flat codebook needs `k=2^B` centroids — capped at `k ≤ n` and intractable past ~16–20 bits. PQ
reaches `B = m·log₂k*` bits at `m·k*` storage. On the **same** 256-d finance cloud, the flat 8-bit
ceiling is recall ≈ 0.53; PQ at 64 bits (8 bytes, 2048 centroids, `2^64` effective codewords)
reaches far higher — a budget no flat codebook can train.

In [ ]:
pq.finance_demo()
pq.test_pq_recall_beats_flat_at_pq_budget()
pq.test_rate_distortion_monotone_pq()

## 6. Independence and the OPQ teaser

PQ assumes the subspaces are independent; cross-subspace correlation and unequal per-subspace
variance (under equal bits) are the loss it pays. A **variance-balancing rotation** lowers
distortion — and a *naive* PCA-align-only rotation barely helps, because it concentrates variance
into one subspace. Finding the rotation that minimizes distortion is exactly **optimized product
quantization**, the next topic. We verify the direction, not a magic number.

In [ ]:
study = pq.rotation_distortion_study(pq.variance_imbalanced_cloud(n=600, d=8, seed=1), 4, 16, seed=1)
for name in ("raw", "random", "pca_only", "balanced"):
    print(f"  PQ distortion ({name:9s}) = {study[name]:.3f}")
pq.test_rotation_lowers_distortion()

## 7. Cross-check and viz constants

Each subspace is just Lloyd, so a sub-codebook matches `scipy.cluster.vq.kmeans2` from the same
init (no faiss). Finally, `viz_constants()` prints the exact toy cloud, sub-codebooks, additive
distortions, scalability frontier, and ADC demo that `ProductQuantizationLaboratory.tsx` bakes.

In [ ]:
pq.validate_against_kmeans2()
print()
pq.viz_constants()

## All claims verified

Every theorem and the honest caveat are executed assertions above. The toy cloud, its two
sub-codebooks, the additive distortions, the scalability frontier, and the ADC demo are the exact
numbers the laboratory mirrors to the decimal. Re-run top to bottom to reproduce the topic.